# Three-bin GEDF tests

This notebook compiles CLASS, runs a full background + perturbation + CMB/matter-spectrum calculation, verifies the three equation-of-state plateaus and both half-transitions, checks energy conservation and finite outputs, confirms removal of the second GEDF component, and tests input validation.

In [2]:
from pathlib import Path
from math import isfinite, log
import subprocess
import tempfile

REPO = Path.cwd()
CONFIG = REPO / 'test' / 'three_bin_gedf.ini'
OUTPUT = REPO / 'test' / 'output'
assert (REPO / 'class').exists() and CONFIG.exists()

In [3]:
build = subprocess.run(['make', 'class'], cwd=REPO, text=True, capture_output=True)
assert build.returncode == 0, build.stdout + build.stderr
print('PASS: compiled CLASS executable')

PASS: compiled CLASS executable


In [5]:
OUTPUT.mkdir(parents=True, exist_ok=True)
run = subprocess.run([str(REPO / 'class'), str(CONFIG)], cwd=REPO, text=True, capture_output=True)
assert run.returncode == 0, run.stdout + run.stderr
print('PASS: full background, perturbation, CMB and P(k) calculation')

PASS: full background, perturbation, CMB and P(k) calculation


In [6]:
def data_rows(path):
    with path.open() as stream:
        for line in stream:
            if line.strip() and not line.startswith('#'):
                yield [float(value) for value in line.split()]

background_path = OUTPUT / 'three_bin_gedf__background.dat'
background = list(data_rows(background_path))
# background columns: z=0, rho_gedf=14, w_gedf=15 (zero based)
def closest_w(target_z):
    row = min(background, key=lambda values: abs(values[0] - target_z))
    return row[0], row[15]

samples = {name: closest_w(z) for name, z in {
    'early': 1e7, 'z_12': 5000.0, 'middle': 1600.0,
    'z_23': 500.0, 'late': 10.0}.items()}
expected = {'early': -1.0, 'z_12': (-1.0 + 1/3)/2,
            'middle': 1/3, 'z_23': (1/3 + 0.0)/2, 'late': 0.0}
tolerance = {'early': 2e-3, 'z_12': 2e-2, 'middle': 2e-3,
             'z_23': 2e-2, 'late': 2e-3}
for name, (_, measured) in samples.items():
    assert abs(measured - expected[name]) < tolerance[name], (name, measured)
samples

{'early': (10004029.43666, -1.0),
 'z_12': (4998.107459256, -0.3319334386913),
 'middle': (1599.774318612, 0.3333120641267),
 'z_23': (500.1497282398, 0.1669427281013),
 'late': (9.997045752212, -5.551115123126e-17)}

In [7]:
# Check d ln(rho_GEDF)/d ln(a) = -3(1+w) through both transitions.
continuity_residuals = []
for previous, current, following in zip(background, background[1:], background[2:]):
    if 1.0 < current[0] < 1e6 and previous[14] > 0 and following[14] > 0:
        a_previous = 1.0 / (1.0 + previous[0])
        a_following = 1.0 / (1.0 + following[0])
        derivative = (log(following[14]) - log(previous[14])) / (log(a_following) - log(a_previous))
        continuity_residuals.append(abs(derivative + 3.0 * (1.0 + current[15])))
max_continuity_residual = max(continuity_residuals)
assert max_continuity_residual < 1e-4
print(f'PASS: background continuity; max residual = {max_continuity_residual:.3e}')

PASS: background continuity; max residual = 8.586e-06


In [8]:
files_to_check = [
    OUTPUT / 'three_bin_gedf__cl_lensed.dat',
    OUTPUT / 'three_bin_gedf__z1_pk.dat',
    OUTPUT / 'three_bin_gedf__z2_pk.dat',
    OUTPUT / 'three_bin_gedf__perturbations_k0_s.dat',
]
counts = {}
for path in files_to_check:
    count = 0
    for row in data_rows(path):
        assert all(isfinite(value) for value in row), path
        count += 1
    assert count > 0
    counts[path.name] = count

perturbation_header = (OUTPUT / 'three_bin_gedf__perturbations_k0_s.dat').read_text().splitlines()[1]
background_header = background_path.read_text().splitlines()[3]
assert 'delta_gedf' in perturbation_header and 'u_gedf' in perturbation_header
assert 'gedf_2' not in perturbation_header.lower() + background_header.lower()
print('PASS: finite spectra and perturbations; single GEDF component only')
counts

PASS: finite spectra and perturbations; single GEDF component only


{'three_bin_gedf__cl_lensed.dat': 1199,
 'three_bin_gedf__z1_pk.dat': 493,
 'three_bin_gedf__z2_pk.dat': 493,
 'three_bin_gedf__perturbations_k0_s.dat': 7285}

In [9]:
invalid_text = CONFIG.read_text().replace('z_12 = 5000.0', 'z_12 = 100.0')
with tempfile.NamedTemporaryFile('w', suffix='.ini') as invalid_file:
    invalid_file.write(invalid_text)
    invalid_file.flush()
    invalid_run = subprocess.run([str(REPO / 'class'), invalid_file.name], cwd=REPO, text=True, capture_output=True)
assert invalid_run.returncode != 0
assert 'z_12 > z_23' in invalid_run.stdout + invalid_run.stderr
print('PASS: invalid transition ordering is rejected')
print('ALL THREE-BIN GEDF TESTS PASSED')

PASS: invalid transition ordering is rejected
ALL THREE-BIN GEDF TESTS PASSED
